In [13]:
! pip install ufal.udpipe

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.8/936.8 kB 4.9 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import os
import re
import subprocess
import nltk
import wget
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem.snowball import SnowballStemmer
from string import punctuation

import pymorphy2
from pymorphy2 import MorphAnalyzer
from gensim.models import Word2Vec, FastText
import gensim.matutils
from gensim.models import KeyedVectors

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

import json


In [2]:
# Загрузка данных NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Загрузка данных
positive_file = 'positive.csv'
negative_file = 'negative.csv'

# Чтение данных
pos = pd.read_csv(positive_file, sep=';', names=['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])
neg = pd.read_csv(negative_file, sep=';', names=['id', 'date', 'name', 'text', 'type', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'])

# Объединение данных
neg['type'] = neg['type'].replace(-1, 0)
df = pd.concat([pos, neg])

# Удаление ненужных столбцов
df = df.drop(['id', 'date', 'name', 'rep', 'rtw', 'faw', 'stcount', 'foll', 'frien', 'listcount'], axis=1)
dfudpipe = df

df


[nltk_data] Downloading package punkt to /home/artyom/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/artyom/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/artyom/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,text,type
0,"@first_timee хоть я и школота, но поверь, у на...",1
1,"Да, все-таки он немного похож на него. Но мой ...",1
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1
...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0
111920,"Вот и в школу, в говно это идти уже надо(",0
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0


In [3]:
stop_words = set(stopwords.words('russian'))

# Инициализация морфологического анализатора
morph = MorphAnalyzer()

# Функция для нормализации текста
def normalize_text(text):
    # Приведение к нижнему регистру
    text = text.lower()
    
    # Удаление всех символов, кроме кириллицы и пробелов
    text = re.sub('[^а-яё ]', '', text)
    
    # Удаление лишних пробелов
    text = re.sub('\\s+', ' ', text).strip()
    
    # Удаление стоп-слов
    words = [word for word in text.split() if word not in stop_words]
    
    # Лемматизация
    words = [morph.parse(word)[0].normal_form for word in words]
    
    return ' '.join(words)

# Применение нормализации к колонке 'text'
df['normalized_text'] = df['text'].apply(normalize_text)

# Токенизация нормализованного текста
df['tokens'] = df['normalized_text'].apply(word_tokenize)


df

,text,type,normalized_text,tokens
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[школотый, поверь, самый, общество, профилиров..."
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[всетаки, немного, похожий, мальчик, равно]"
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[идиотка, испугаться]"
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[угол, сидеть, погибать, голод, ещё, порция, в..."
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[значит, страшилка, блинпосмотреть, частиу, со..."
...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]"
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]"
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]"
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]"


In [4]:
# Загрузка тонального словаря
tonality_dict = pd.read_csv('dictionary_documents/words_all_full_rating.csv', encoding="cp1251", sep=';')
tonality_dict['mean'] = tonality_dict['mean'].str.replace(',', '.').astype(float)
tonality_dict = dict(zip(tonality_dict['Words'], tonality_dict['mean']))

# Функция для извлечения тональных признаков
def extract_tonality_features(tokens):
    # Получаем тональные значения для токенов, которые есть в словаре
    tonality_scores = [tonality_dict.get(token, 0) for token in tokens if token in tonality_dict]
    
    # Если список пуст, возвращаем [0, 0, 0, 0, 0, 0]
    if not tonality_scores:
        return [0, 0, 0, 0, 0, 0]
    
    # Вычисляем признаки
    mean_tonality = sum(tonality_scores) / len(tonality_scores)  # Среднее значение
    max_tonality = max(tonality_scores)  # Максимальное значение
    min_tonality = min(tonality_scores)  # Минимальное значение
    sum_tonality = sum(tonality_scores)  # Суммарное значение
    pos_count = len([score for score in tonality_scores if score > 0])  # Количество положительных значений
    neg_count = len([score for score in tonality_scores if score < 0])  # Количество отрицательных значений
    
    return [mean_tonality, max_tonality, min_tonality, sum_tonality, pos_count, neg_count]

# Применение функции к токенам
df['tonality_features'] = df['tokens'].apply(extract_tonality_features)

# Разделение списка признаков на отдельные колонки
df[['tonality_mean', 'tonality_max', 'tonality_min', 'tonality_sum', 'tonality_pos_count', 'tonality_neg_count']] = pd.DataFrame(df['tonality_features'].tolist(), index=df.index)

# Удаление временной колонки с признаками
df.drop(columns=['tonality_features'], inplace=True)

# Сохранение результатов в файл
df.to_csv('text_with_tonality_features.csv', index=False)

df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[школотый, поверь, самый, общество, профилиров...",0.083333,0.333333,0.000000,0.333333,1,0
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[всетаки, немного, похожий, мальчик, равно]",0.000000,0.000000,0.000000,0.000000,0,0
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[идиотка, испугаться]",-0.833333,-0.833333,-0.833333,-0.833333,0,1
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[угол, сидеть, погибать, голод, ещё, порция, в...",-0.511111,0.000000,-1.333333,-1.533333,0,2
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[значит, страшилка, блинпосмотреть, частиу, со...",0.000000,0.000000,0.000000,0.000000,0,0
...,...,...,...,...,...,...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0


In [5]:
# Инициализация морфологического анализатора
morph = MorphAnalyzer()

# Функция для извлечения морфологических признаков
def extract_morph_features(tokens):
    # Словарь для подсчета частей речи
    pos_counts = {'NOUN': 0, 'VERB': 0, 'ADJ': 0, 'ADV': 0, 'OTHER': 0}
    
    # Подсчет частей речи
    for token in tokens:
        parsed_word = morph.parse(token)[0]  # Анализ токена
        pos = parsed_word.tag.POS  # Получение части речи
        
        # Увеличиваем счетчик для соответствующей части речи
        if pos in pos_counts:
            pos_counts[pos] += 1
        else:
            pos_counts['OTHER'] += 1
    
    # Вычисление относительной частоты
    total_words = len(tokens)
    if total_words == 0:
        return [0, 0, 0, 0, 0]  # Если токенов нет, возвращаем нули
    
    # Возвращаем относительные частоты
    return [
        pos_counts['NOUN'] / total_words,  # Доля существительных
        pos_counts['VERB'] / total_words,  # Доля глаголов
        pos_counts['ADJ'] / total_words,   # Доля прилагательных
        pos_counts['ADV'] / total_words,   # Доля наречий
        pos_counts['OTHER'] / total_words  # Доля других частей речи
    ]

# Применение функции к токенам
df['morph_features'] = df['tokens'].apply(extract_morph_features)

# Разделение списка признаков на отдельные колонки
df[['noun_freq', 'verb_freq', 'adj_freq', 'adv_freq', 'other_freq']] = pd.DataFrame(df['morph_features'].tolist(), index=df.index)

# Удаление временной колонки с признаками
df.drop(columns=['morph_features'], inplace=True)

# Сохранение результатов в файл
df.to_csv('text_with_morph_features.csv', index=False)

df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,noun_freq,verb_freq,adj_freq,adv_freq,other_freq
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[школотый, поверь, самый, общество, профилиров...",0.083333,0.333333,0.000000,0.333333,1,0,0.428571,0.0,0.0,0.0,0.571429
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[всетаки, немного, похожий, мальчик, равно]",0.000000,0.000000,0.000000,0.000000,0,0,0.400000,0.0,0.0,0.0,0.600000
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[идиотка, испугаться]",-0.833333,-0.833333,-0.833333,-0.833333,0,1,0.500000,0.0,0.0,0.0,0.500000
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[угол, сидеть, погибать, голод, ещё, порция, в...",-0.511111,0.000000,-1.333333,-1.533333,0,2,0.300000,0.0,0.0,0.0,0.700000
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[значит, страшилка, блинпосмотреть, частиу, со...",0.000000,0.000000,0.000000,0.000000,0,0,0.333333,0.0,0.0,0.0,0.666667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.0,0.0,0.0,1.000000
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0,0.200000,0.0,0.0,0.0,0.800000
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0,0.500000,0.0,0.0,0.0,0.500000
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0,0.666667,0.0,0.0,0.0,0.333333


In [6]:
# Инициализация TfidfVectorizer с ограничением на 1000 признаков
vectorizer = TfidfVectorizer(max_features=1000)

# Преобразование текста в TF-IDF признаки
tfidf_features = vectorizer.fit_transform(df['normalized_text']).toarray()

# Создание DataFrame с TF-IDF признаками
tfidf_df = pd.DataFrame(tfidf_features, columns=[f'tfidf_{i}' for i in range(tfidf_features.shape[1])])

# Сбросить индексы в обоих DataFrame
df = df.reset_index(drop=True)
tfidf_df = tfidf_df.reset_index(drop=True)

# Объединение TF-IDF признаков с исходным DataFrame
df = pd.concat([df, tfidf_df], axis=1)

# Сохранение результатов в файл
df.to_csv('text_with_tfidf_features.csv', index=False)

df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,tfidf_990,tfidf_991,tfidf_992,tfidf_993,tfidf_994,tfidf_995,tfidf_996,tfidf_997,tfidf_998,tfidf_999
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[школотый, поверь, самый, общество, профилиров...",0.083333,0.333333,0.000000,0.333333,1,0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[всетаки, немного, похожий, мальчик, равно]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[идиотка, испугаться]",-0.833333,-0.833333,-0.833333,-0.833333,0,1,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[угол, сидеть, погибать, голод, ещё, порция, в...",-0.511111,0.000000,-1.333333,-1.533333,0,2,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[значит, страшилка, блинпосмотреть, частиу, со...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226830,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226831,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.317762,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226832,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Обучение модели Word2Vec
word2vec_model = Word2Vec(
    sentences=df['tokens'],  # Токенизированные тексты
    vector_size=300,         # Размерность векторов
    window=5,                # Размер окна (сколько слов вокруг учитывать)
    min_count=1,             # Минимальная частота слова для учета
    workers=4                # Количество потоков для обучения
)

# Функция для извлечения среднего вектора текста
def get_text_vector(tokens, model):
    # Фильтруем слова, которые есть в модели
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    
    # Если в тексте нет слов из модели, возвращаем нулевой вектор
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    # Вычисляем средний вектор
    return np.mean(vectors, axis=0)

# Применение функции к каждому тексту
word2vec_features = np.array([get_text_vector(tokens, word2vec_model) for tokens in df['tokens']])

# Создание DataFrame с Word2Vec-признаками
word2vec_df = pd.DataFrame(word2vec_features, columns=[f'w2v_{i}' for i in range(word2vec_features.shape[1])])

# Объединение Word2Vec-признаков с исходным DataFrame
df = pd.concat([df, word2vec_df], axis=1)

# Сохранение результатов в файл
df.to_csv('text_with_word2vec_features.csv', index=False)

df

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,w2v_290,w2v_291,w2v_292,w2v_293,w2v_294,w2v_295,w2v_296,w2v_297,w2v_298,w2v_299
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[школотый, поверь, самый, общество, профилиров...",0.083333,0.333333,0.000000,0.333333,1,0,...,0.168842,0.251940,0.378577,0.043225,0.385851,0.672249,-0.113136,-0.132460,0.226311,-0.164363
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[всетаки, немного, похожий, мальчик, равно]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.393395,0.400120,0.457106,-0.072121,0.789036,0.865912,-0.187938,-0.465102,0.401329,-0.256225
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[идиотка, испугаться]",-0.833333,-0.833333,-0.833333,-0.833333,0,1,...,0.121677,0.120228,0.115001,-0.010261,0.205700,0.224045,-0.017232,-0.087288,0.114703,-0.075400
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[угол, сидеть, погибать, голод, ещё, порция, в...",-0.511111,0.000000,-1.333333,-1.533333,0,2,...,0.372192,0.541125,0.488986,0.185707,0.460446,0.466311,-0.110837,-0.159916,0.203865,-0.220014
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[значит, страшилка, блинпосмотреть, частиу, со...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.179853,0.210451,0.215461,0.000835,0.315558,0.368450,-0.012074,-0.153031,0.122344,-0.152629
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[каждый, хотеть, исправлять]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.328608,0.573468,0.655895,-0.020275,0.568387,0.621090,-0.298100,-0.434475,0.201394,-0.154549
226830,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать, вправлять, мозг, равно, скучать]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.174090,0.515659,0.614758,0.042479,0.606224,0.774592,-0.185293,-0.144850,0.292274,-0.371510
226831,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[школа, говно, это, идти]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.477070,0.496568,1.048619,0.085533,0.882818,0.725270,-0.242937,-0.446867,0.488379,-0.306980
226832,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[тауриэль, грусть, обнять]",0.000000,0.000000,0.000000,0.000000,0,0,...,0.141389,0.177935,0.208353,0.040752,0.308546,0.361975,0.032502,-0.122343,0.168744,-0.166469


In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Загрузка данных
df = pd.read_csv('text_with_word2vec_features.csv')

# Разделение данных на признаки (X) и целевую переменную (y)
X = df.drop(columns=['text', 'type', 'normalized_text', 'tokens'])  # Убираем ненужные колонки
y = df['type']  # Целевая переменная

# Разделение на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Создание и обучение модели
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Предсказание на тестовой выборке
y_pred = model.predict(X_test)

# Вычисление метрик
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Вывод результатов
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# Отчет о классификации
report = classification_report(y_test, y_pred)
print(report)

Precision: 0.6605
Recall: 0.7313
F1-score: 0.6941
              precision    recall  f1-score   support

           0       0.69      0.62      0.65     22480
           1       0.66      0.73      0.69     22887

    accuracy                           0.67     45367
   macro avg       0.68      0.67      0.67     45367
weighted avg       0.68      0.67      0.67     45367



/home/artyom/itmo_labs/itmo/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [9]:
import subprocess
import os

def preprocess_text_with_udpipe(text):
    try:
        # Сохраняем текст во временный файл
        with open('temp_input.txt', 'w', encoding='utf-8') as f:
            f.write(text)
        
        # Вызов скрипта UDPipe
        result = subprocess.run([
            'python3', 'rus_preprocessing_udpipe.py',
            '--input', 'temp_input.txt',
            '--output', 'temp_output.txt'
        ], capture_output=True, text=True)
        
        # Проверка, завершился ли скрипт успешно
        if result.returncode != 0:
            raise RuntimeError(f"Скрипт завершился с ошибкой: {result.stderr}")
        
        # Проверка, создан ли выходной файл
        if not os.path.exists('temp_output.txt'):
            raise FileNotFoundError("Выходной файл не был создан.")
        
        # Чтение результата
        with open('temp_output.txt', 'r', encoding='utf-8') as f:
            processed_text = f.read()
        
        return processed_text
    
    except Exception as e:
        print(f"Ошибка при обработке текста: {e}")
        return text  # Возвращаем исходный текст в случае ошибки

In [14]:
from ufal.udpipe import Model, Pipeline
import pandas as pd

# Загрузка модели UDPipe (один раз)
model_path = "udpipe_syntagrus.model"  # Укажите путь к модели
model = Model.load(model_path)
if not model:
    raise Exception("Не удалось загрузить модель UDPipe.")
pipeline = Pipeline(model, "tokenize", Pipeline.DEFAULT, Pipeline.DEFAULT, "conllu")

# Функция для обработки текста с использованием UDPipe
def preprocess_text_with_udpipe(text):
    processed = pipeline.process(text)
    # Извлечение лемм и частей речи (пример из вашего скрипта)
    content = [line for line in processed.split("\n") if not line.startswith("#")]
    tagged = [w.split("\t") for w in content if w]
    lemmas = []
    for t in tagged:
        if len(t) != 10:
            continue
        lemma = t[2]  # Лемма
        pos = t[3]    # Часть речи
        lemmas.append(f"{lemma}_{pos}")
    return " ".join(lemmas)

# Применение функции к каждому тексту в dfudpipe
dfudpipe['processed_text'] = dfudpipe['text'].apply(preprocess_text_with_udpipe)

# Токенизация предобработанного текста
dfudpipe['tokens'] = dfudpipe['processed_text'].apply(lambda x: x.split())

dfudpipe

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,noun_freq,verb_freq,adj_freq,adv_freq,other_freq,processed_text
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[@first_timee_X, хоть_ADV, я_PRON, и_PART, шка...",0.083333,0.333333,0.000000,0.333333,1,0,0.428571,0.0,0.0,0.0,0.571429,@first_timee_X хоть_ADV я_PRON и_PART шкалыват...
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[да_PART, ,_PUNCT, все-таки_PART, он_PRON, нем...",0.000000,0.000000,0.000000,0.000000,0,0,0.400000,0.0,0.0,0.0,0.600000,"да_PART ,_PUNCT все-таки_PART он_PRON немного_..."
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[Rt_PROPN, @KatiaCheh_PROPN, :_PUNCT, ну_PART,...",-0.833333,-0.833333,-0.833333,-0.833333,0,1,0.500000,0.0,0.0,0.0,0.500000,Rt_PROPN @KatiaCheh_PROPN :_PUNCT ну_PART ты_P...
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[Rt_PROPN, @digger2912_NOUN, :_PUNCT, ""_PUNCT,...",-0.511111,0.000000,-1.333333,-1.533333,0,2,0.300000,0.0,0.0,0.0,0.700000,"Rt_PROPN @digger2912_NOUN :_PUNCT ""_PUNCT кто_..."
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[@irina_dyshkant_X, вот_PART, что_PRON, значит...",0.000000,0.000000,0.000000,0.000000,0,0,0.333333,0.0,0.0,0.0,0.666667,@irina_dyshkant_X вот_PART что_PRON значить_VE...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111918,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[но_CCONJ, не_PART, каждый_ADJ, хотеть_VERB, ч...",0.000000,0.000000,0.000000,0.000000,0,0,0.000000,0.0,0.0,0.0,1.000000,но_CCONJ не_PART каждый_ADJ хотеть_VERB что_PR...
111919,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать_VERB, так_ADV, :_PUNCT, -_PUNCT, (_PU...",0.000000,0.000000,0.000000,0.000000,0,0,0.200000,0.0,0.0,0.0,0.800000,скучать_VERB так_ADV :_PUNCT -_PUNCT (_PUNCT т...
111920,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[вот_PART, и_PART, в_ADP, школа_NOUN, ,_PUNCT,...",0.000000,0.000000,0.000000,0.000000,0,0,0.500000,0.0,0.0,0.0,0.500000,"вот_PART и_PART в_ADP школа_NOUN ,_PUNCT в_ADP..."
111921,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[Rt_PROPN, @_Them___NOUN, :_PUNCT, @LisaBeroud...",0.000000,0.000000,0.000000,0.000000,0,0,0.666667,0.0,0.0,0.0,0.333333,Rt_PROPN @_Them___NOUN :_PUNCT @LisaBeroud_PRO...


In [15]:
# Сохранение результата в CSV-файл
output_file = "processed_texts.csv"  # Имя выходного файла
dfudpipe.to_csv(output_file, index=False, encoding='utf-8')

print(f"Результат сохранен в файл: {output_file}")

Результат сохранен в файл: processed_texts.csv


In [19]:
from gensim.models import KeyedVectors

# 3. Загрузка предобученной модели Word2Vec
word2vec_model_path = "180/model.bin"  # Укажите путь к модели Word2Vec
word2vec_model = KeyedVectors.load_word2vec_format(word2vec_model_path, binary=True)

# Функция для получения среднего вектора текста
def get_text_vector(tokens, model):
    vectors = []
    for token in tokens:
        if token in model:  # Проверяем, есть ли слово в модели
            vectors.append(model[token])
    if len(vectors) == 0:
        return np.zeros(model.vector_size)  # Возвращаем нулевой вектор, если слов нет в модели
    return np.mean(vectors, axis=0)  # Усредняем векторы

# Применение функции к предобработанным текстам
dfudpipe['word2vec_vector'] = dfudpipe['tokens'].apply(lambda x: get_text_vector(x, word2vec_model))

# 4. Преобразование векторов в отдельные столбцы
word2vec_columns = pd.DataFrame(dfudpipe['word2vec_vector'].tolist(), columns=[f'w2v_{i}' for i in range(word2vec_model.vector_size)])

# Сброс индекса для dfudpipe и word2vec_columns
dfudpipe = dfudpipe.reset_index(drop=True)
word2vec_columns = word2vec_columns.reset_index(drop=True)

# Объединение с исходным DataFrame
dfudpipe = pd.concat([dfudpipe, word2vec_columns], axis=1)

# 5. Сохранение результата в CSV
output_file = "texts_with_word2vec_vectors.csv"
dfudpipe.to_csv(output_file, index=False, encoding='utf-8')

print(f"Результат сохранен в файл: {output_file}")

dfudpipe

Результат сохранен в файл: texts_with_word2vec_vectors.csv


,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,w2v_290,w2v_291,w2v_292,w2v_293,w2v_294,w2v_295,w2v_296,w2v_297,w2v_298,w2v_299
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[@first_timee_X, хоть_ADV, я_PRON, и_PART, шка...",0.083333,0.333333,0.000000,0.333333,1,0,...,-0.302308,-0.682788,-0.764171,0.566921,0.513510,-0.100435,-0.476615,0.506071,1.194629,-0.508285
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[да_PART, ,_PUNCT, все-таки_PART, он_PRON, нем...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.424478,-1.582956,-0.597059,0.224225,0.339300,0.399364,-0.825179,-0.884485,-0.061881,0.364045
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[Rt_PROPN, @KatiaCheh_PROPN, :_PUNCT, ну_PART,...",-0.833333,-0.833333,-0.833333,-0.833333,0,1,...,1.843332,-0.131286,0.362703,-0.076507,-1.453561,0.379717,-1.450538,-0.485124,-0.556431,0.092679
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[Rt_PROPN, @digger2912_NOUN, :_PUNCT, ""_PUNCT,...",-0.511111,0.000000,-1.333333,-1.533333,0,2,...,0.389059,-0.907503,-0.522165,0.082773,-1.206338,0.399251,0.077377,0.428756,-0.497550,0.666755
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[@irina_dyshkant_X, вот_PART, что_PRON, значит...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.146253,0.325114,-0.703294,0.078491,-0.676953,0.293301,-0.764982,-0.349880,-0.022869,0.012185
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[но_CCONJ, не_PART, каждый_ADJ, хотеть_VERB, ч...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.083337,-0.409721,-0.569859,0.571803,1.187466,-0.279271,-0.121722,1.041201,-0.338638,1.303418
226830,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать_VERB, так_ADV, :_PUNCT, -_PUNCT, (_PU...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.173991,-0.856615,-0.109176,0.607033,0.411901,0.904253,-0.388797,0.360963,1.181812,-0.743449
226831,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[вот_PART, и_PART, в_ADP, школа_NOUN, ,_PUNCT,...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.288305,-1.031953,-1.696009,0.485535,0.605386,0.620900,0.885101,0.502123,-0.759945,0.701073
226832,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[Rt_PROPN, @_Them___NOUN, :_PUNCT, @LisaBeroud...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.478570,-0.125201,-0.081201,0.170761,-0.414099,0.302467,-0.312895,0.025542,0.139697,-0.159138


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# 1. Подготовка данных
# X - признаки (Word2Vec-векторы), y - целевая переменная (type)
X = dfudpipe[[f'w2v_{i}' for i in range(word2vec_model.vector_size)]]
y = dfudpipe['type']

# 2. Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Обучение модели (например, Random Forest)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 4. Предсказание на тестовой выборке
y_pred = model.predict(X_test)

# 5. Оценка модели
# Вывод отчета о классификации
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Отдельные метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.66      0.67     22480
           1       0.68      0.70      0.69     22887

    accuracy                           0.68     45367
   macro avg       0.68      0.68      0.68     45367
weighted avg       0.68      0.68      0.68     45367

Accuracy: 0.6780
Precision: 0.6752
Recall: 0.6970
F1-Score: 0.6859


In [21]:
from gensim.models import FastText
import numpy as np

# 1. Обучение FastText-модели на ваших данных
fasttext_model = FastText(sentences=dfudpipe['tokens'], vector_size=100, window=5, min_count=1, workers=4)

# 2. Функция для усреднения векторов слов в тексте
def get_fasttext_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)  # Возвращаем нулевой вектор, если слов нет в модели
    return np.mean(vectors, axis=0)  # Усредняем векторы

# 3. Применение функции к каждому тексту
dfudpipe['fasttext_vector'] = dfudpipe['tokens'].apply(lambda x: get_fasttext_vector(x, fasttext_model))

# 4. Преобразование векторов в отдельные столбцы
fasttext_columns = pd.DataFrame(dfudpipe['fasttext_vector'].tolist(), columns=[f'ft_{i}' for i in range(fasttext_model.vector_size)])

# 5. Объединение с исходным DataFrame
dfudpipe = pd.concat([dfudpipe, fasttext_columns], axis=1)

# 6. Сохранение результата
dfudpipe.to_csv("texts_with_fasttext_vectors.csv", index=False, encoding='utf-8')
print("FastText-векторы сохранены в texts_with_fasttext_vectors.csv")

dfudpipe

FastText-векторы сохранены в texts_with_fasttext_vectors.csv


,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,ft_90,ft_91,ft_92,ft_93,ft_94,ft_95,ft_96,ft_97,ft_98,ft_99
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[@first_timee_X, хоть_ADV, я_PRON, и_PART, шка...",0.083333,0.333333,0.000000,0.333333,1,0,...,0.104422,-1.552266,0.291208,0.511182,0.281179,-1.544000,-0.346879,-1.948437,-1.830339,0.831780
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[да_PART, ,_PUNCT, все-таки_PART, он_PRON, нем...",0.000000,0.000000,0.000000,0.000000,0,0,...,1.010986,-0.702725,0.779978,-0.060284,0.405742,-1.683718,-0.241738,-1.415357,-1.838045,0.825374
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[Rt_PROPN, @KatiaCheh_PROPN, :_PUNCT, ну_PART,...",-0.833333,-0.833333,-0.833333,-0.833333,0,1,...,0.394398,0.348982,0.696732,-0.387193,-0.364995,-0.276497,0.372486,-1.850445,-2.557389,1.396685
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[Rt_PROPN, @digger2912_NOUN, :_PUNCT, ""_PUNCT,...",-0.511111,0.000000,-1.333333,-1.533333,0,2,...,0.288281,-1.455747,0.992856,0.155296,0.289752,-1.628173,-0.498880,-1.120272,-1.538643,0.553676
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[@irina_dyshkant_X, вот_PART, что_PRON, значит...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.573658,-1.159069,0.161470,0.697701,0.086714,-1.422976,-0.423697,-1.072206,-1.662464,1.662675
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[но_CCONJ, не_PART, каждый_ADJ, хотеть_VERB, ч...",0.000000,0.000000,0.000000,0.000000,0,0,...,1.227058,-1.604622,1.076323,0.059827,0.379730,-1.661891,0.143396,-1.383338,-2.621821,1.178018
226830,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[скучать_VERB, так_ADV, :_PUNCT, -_PUNCT, (_PU...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.489640,-1.417743,0.583246,0.016835,0.840747,-0.946194,-0.280104,-1.785901,-1.882216,1.566854
226831,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[вот_PART, и_PART, в_ADP, школа_NOUN, ,_PUNCT,...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.062751,-2.092139,0.856622,-0.208249,-0.540093,-2.369535,-0.990047,-1.908860,-1.972073,-0.030184
226832,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[Rt_PROPN, @_Them___NOUN, :_PUNCT, @LisaBeroud...",0.000000,0.000000,0.000000,0.000000,0,0,...,-0.165254,0.292345,0.347053,0.450553,0.484537,-0.103160,-0.440058,-0.757136,-1.323784,1.916642


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

# 1. Подготовка данных
# X - FastText-векторы, y - целевая переменная (type)
X = dfudpipe[[f'ft_{i}' for i in range(fasttext_model.vector_size)]]
y = dfudpipe['type']

# 2. Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Обучение модели (например, Random Forest)
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# 4. Предсказание на тестовой выборке
y_pred = model.predict(X_test)

# 5. Оценка модели
# Вывод отчета о классификации
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Отдельные метрики
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.86      0.87     22480
           1       0.87      0.88      0.87     22887

    accuracy                           0.87     45367
   macro avg       0.87      0.87      0.87     45367
weighted avg       0.87      0.87      0.87     45367

Accuracy: 0.8694
Precision: 0.8662
Recall: 0.8767
F1-Score: 0.8714


In [24]:
import numpy as np
import pandas as pd
from gensim.models import KeyedVectors

# Загрузка данных из processed_texts.csv
processed_texts_path = 'processed_texts.csv'  # Укажите путь к файлу
dfft = pd.read_csv(processed_texts_path)

# Проверка структуры данных
print(dfft.head())  # Убедимся, что данные загружены правильно

# Загрузка предобученной модели FastText
fasttext_model_path = '180/model.bin'  # Укажите путь к скачанной модели
fasttext_model = KeyedVectors.load_word2vec_format(fasttext_model_path, binary=True)

# Функция для извлечения среднего вектора текста с использованием предобученной модели FastText
def get_fasttext_vector(tokens, model):
    # Фильтруем слова, которые есть в модели
    vectors = [model[word] for word in tokens if word in model]
    
    # Если в тексте нет слов из модели, возвращаем нулевой вектор
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    
    # Вычисляем средний вектор
    return np.mean(vectors, axis=0)

# Предположим, что в dfft есть столбец 'tokens' с токенизированными текстами
# Если токены хранятся в виде строки, преобразуем их в список
if isinstance(dfft['tokens'].iloc[0], str):
    dfft['tokens'] = dfft['tokens'].apply(lambda x: x.split())

# Применение функции к каждому тексту
fasttext_features = np.array([get_fasttext_vector(tokens, fasttext_model) for tokens in dfft['tokens']])

# Создание DataFrame с FastText-признаками
fasttext_df = pd.DataFrame(fasttext_features, columns=[f'fasttext_{i}' for i in range(fasttext_features.shape[1])])

# Объединение FastText-признаков с исходным DataFrame
dfft = pd.concat([dfft, fasttext_df], axis=1)

# Сохранение результатов в файл
dfft.to_csv('text_with_fasttext_features.csv', index=False)

# Вывод результата
print(dfft.head())

                                                text  type  \
0  @first_timee хоть я и школота, но поверь, у на...     1   
1  Да, все-таки он немного похож на него. Но мой ...     1   
2  RT @KatiaCheh: Ну ты идиотка) я испугалась за ...     1   
3  RT @digger2912: "Кто то в углу сидит и погибае...     1   
4  @irina_dyshkant Вот что значит страшилка :D\nН...     1   

                                     normalized_text  \
0  школотый поверь самый общество профилировать п...   
1              всетаки немного похожий мальчик равно   
2                                 идиотка испугаться   
3  угол сидеть погибать голод ещё порция взять хо...   
4  значит страшилка блинпосмотреть частиу создать...   

                                              tokens  tonality_mean  \
0  ['@first_timee_X', 'хоть_ADV', 'я_PRON', 'и_PA...       0.083333   
1  ['да_PART', ',_PUNCT', 'все-таки_PART', 'он_PR...       0.000000   
2  ['Rt_PROPN', '@KatiaCheh_PROPN', ':_PUNCT', 'н...      -0.833333   
3  ['R

In [25]:
dfft

,text,type,normalized_text,tokens,tonality_mean,tonality_max,tonality_min,tonality_sum,tonality_pos_count,tonality_neg_count,...,fasttext_290,fasttext_291,fasttext_292,fasttext_293,fasttext_294,fasttext_295,fasttext_296,fasttext_297,fasttext_298,fasttext_299
0,"@first_timee хоть я и школота, но поверь, у на...",1,школотый поверь самый общество профилировать п...,"[['@first_timee_X',, 'хоть_ADV',, 'я_PRON',, '...",0.083333,0.333333,0.000000,0.333333,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,"Да, все-таки он немного похож на него. Но мой ...",1,всетаки немного похожий мальчик равно,"[['да_PART',, ',_PUNCT',, 'все-таки_PART',, 'о...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,RT @KatiaCheh: Ну ты идиотка) я испугалась за ...,1,идиотка испугаться,"[['Rt_PROPN',, '@KatiaCheh_PROPN',, ':_PUNCT',...",-0.833333,-0.833333,-0.833333,-0.833333,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,"RT @digger2912: ""Кто то в углу сидит и погибае...",1,угол сидеть погибать голод ещё порция взять хо...,"[['Rt_PROPN',, '@digger2912_NOUN',, ':_PUNCT',...",-0.511111,0.000000,-1.333333,-1.533333,0,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,@irina_dyshkant Вот что значит страшилка :D\nН...,1,значит страшилка блинпосмотреть частиу создать...,"[['@irina_dyshkant_X',, 'вот_PART',, 'что_PRON...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226829,Но не каждый хочет что то исправлять:( http://...,0,каждый хотеть исправлять,"[['но_CCONJ',, 'не_PART',, 'каждый_ADJ',, 'хот...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226830,скучаю так :-( только @taaannyaaa вправляет мо...,0,скучать вправлять мозг равно скучать,"[['скучать_VERB',, 'так_ADV',, ':_PUNCT',, '-_...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226831,"Вот и в школу, в говно это идти уже надо(",0,школа говно это идти,"[['вот_PART',, 'и_PART',, 'в_ADP',, 'школа_NOU...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226832,"RT @_Them__: @LisaBeroud Тауриэль, не грусти :...",0,тауриэль грусть обнять,"[['Rt_PROPN',, '@_Them___NOUN',, ':_PUNCT',, '...",0.000000,0.000000,0.000000,0.000000,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Убедимся, что в X только числовые признаки
# Удаляем все нечисловые столбцы, такие как 'text', 'tokens', 'normalized_text'
X = dfft.drop(columns=['text', 'tokens', 'normalized_text', 'type', 'processed_text'])  # Удаляем текстовые столбцы
y = dfft['type']  # Целевая переменная

# Проверка типов данных в X
print("Типы данных в X:")
print(X.dtypes)

# Проверка первых строк X
print("\nПервые строки X:")
print(X.head())

# Проверка на пропущенные значения
print("\nПропущенные значения в X:")
print(X.isnull().sum())

# Разделим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Инициализация и обучение модели (например, Logistic Regression)
model = LogisticRegression(max_iter=1000)  # Увеличиваем max_iter для сходимости
model.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_pred = model.predict(X_test)

# Оценка метрик
precision = precision_score(y_test, y_pred, average='binary')  # Для бинарной классификации
recall = recall_score(y_test, y_pred, average='binary')
f1 = f1_score(y_test, y_pred, average='binary')

# Вывод метрик
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

# Детальный отчёт по метрикам
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Class 0', 'Class 1']))

Типы данных в X:
tonality_mean         float64
tonality_max          float64
tonality_min          float64
tonality_sum          float64
tonality_pos_count      int64
                       ...   
fasttext_295          float64
fasttext_296          float64
fasttext_297          float64
fasttext_298          float64
fasttext_299          float64
Length: 311, dtype: object

Первые строки X:
   tonality_mean  tonality_max  tonality_min  tonality_sum  \
0       0.083333      0.333333      0.000000      0.333333   
1       0.000000      0.000000      0.000000      0.000000   
2      -0.833333     -0.833333     -0.833333     -0.833333   
3      -0.511111      0.000000     -1.333333     -1.533333   
4       0.000000      0.000000      0.000000      0.000000   

   tonality_pos_count  tonality_neg_count  noun_freq  verb_freq  adj_freq  \
0                   1                   0   0.428571        0.0       0.0   
1                   0                   0   0.400000        0.0       0.0   
2   

# F1
word2vec мой     0.67
word2vec чужой   0.68
fasttext мой     0.87
fasttext чужой   0.57